# Projeto de Fine-Tuning de LLM para Atendimento Médico Virtual
Este notebook executa o pipeline completo de especialização do modelo Llama-3 (8b) utilizando dados médicos reais do dataset MedQuAD. O processo engloba desde o download da base até o treinamento otimizado via Unsloth e salvamento do adaptador LoRA.

## 1. Instalação das ferramentas necessárias para o download dos dados e do framework Unsloth

In [ ]:
!pip install kagglehub[pandas-datasets]

### 2. Ingestão e Exploração dos Dados (Camada Bronze)
Download do dataset MedQuAD via API do Kaggle e carregamento em um DataFrame Pandas para análise preliminar das colunas estruturadas de perguntas e respostas médicas.

In [ ]:
import kagglehub
import os
import pandas as pd

# Baixa o dataset
path = kagglehub.dataset_download("pythonafroz/medquad-medical-question-answer-for-ai-research")

arquivos = os.listdir(path)

# Buscando o primeiro arquivo .csv da pasta
arquivo_csv = [f for f in arquivos if f.endswith('.csv')][0]
full_path = os.path.join(path, arquivo_csv)

# Criando dataframe pandas o arquivo
df_bronze = pd.read_csv(full_path)

df_bronze.describe()

Using Colab cache for faster access to the 'medquad-medical-question-answer-for-ai-research' dataset.


In [ ]:
display(df_bronze)

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
...,...,...,...,...
16407,What is (are) Diabetic Neuropathies: The Nerve...,Focal neuropathy appears suddenly and affects ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16408,How to prevent Diabetic Neuropathies: The Nerv...,The best way to prevent neuropathy is to keep ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16409,How to diagnose Diabetic Neuropathies: The Ner...,Doctors diagnose neuropathy on the basis of sy...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16410,What are the treatments for Diabetic Neuropath...,The first treatment step is to bring blood glu...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...


### 3. Pré-processamento e Limpeza Base (Camada Silver)
Tratamento e higienização das strings utilizando expressões regulares (Regex). Esta etapa remove ruídos, textos padrões repetitivos de páginas web (como avisos de vídeos e dicionários) e formata o espaçamento das perguntas e respostas.

In [ ]:
# 1. Filtrar nulos
df_limpo = df_bronze.dropna(subset=['question', 'answer']).copy()

# 2. Removendo colunas desnecessárias
df_limpo = df_limpo.drop(columns=['source', 'focus_area'], errors='ignore')

# Removendo textos inúteis para o fine-tuning
textos_para_remover = [
    r"The Human Phenotype Ontology provides the following list of signs and symptoms for",
    r"You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms\.",
    r"To enlarge the video, click the brackets in the lower right-hand corner\.",
    r"To reduce the video, press the Escape \(Esc\) button on your keyboard\.",
    r"See this graphic for a quick overview.*",
    r"Get tips on finding an eye care professional\.",
    r"Learn what a comprehensive dilated eye exam involves\.",
    r"To enlarge the videos on this page, click the brackets in the lower right-hand corner of the video screen\.",
    r"To reduce the videos, press the Escape \(Esc\) button on your keyboard\.\)",
    r"\(Watch the( animated)? video to learn more about.*?\)"
]

# 3. Pré-processamento: Loop de Limpeza
for texto_sujo in textos_para_remover:
    df_limpo['answer'] = df_limpo['answer'].str.replace(texto_sujo, '', regex=True)

# 4. Limpeza final de espaços e trim
df_limpo['question'] = df_limpo['question'].str.replace(r'\s+', ' ', regex=True).str.strip()
df_limpo['answer'] = df_limpo['answer'].str.replace(r'\s+', ' ', regex=True).str.strip()

# 5. Correção do espaço antes da interrogação
df_limpo['question'] = df_limpo['question'].str.replace(r'\s+\?', '?', regex=True)

display(df_limpo)

### 4. Configuração do Ambiente do Modelo
Instalação e atualização dos pacotes fundamentais do ecossistema Hugging Face, PEFT (LoRA) e a biblioteca de aceleração de hardware Unsloth.

In [ ]:
!pip install --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-deps xformers peft accelerate bitsandbytes trl
!pip install --upgrade transformers datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-6zmv0ufl/unsloth_65fb0d7085474c8fa841da9fbd0a9995
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-6zmv0ufl/unsloth_65fb0d7085474c8fa841da9fbd0a9995
  Resolved https://github.com/unslothai/unsloth.git to commit eeb49d54b8d801a1ce922fa15f778e2bad205db0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached datasets-4.3.0-py3-none-any.whl (506 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Suc

### 5. Estruturação do Dataset no Padrão Alpaca (Camada Gold)
Mapeamento dos dados limpos para as colunas `instruction`, `input` e `output`, gerando o formato ideal para que o LLM aprenda a seguir instruções de forma assertiva e conversacional.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
import json
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset


# Criação da estrutura do prompt Alpaca diretamente nas colunas
df_limpo['instruction'] = "You are a medical virtual assistant. Answer the following medical question accurately."
df_limpo['input'] = df_limpo['question']
df_limpo['output'] = df_limpo['answer']

# Filtra apenas as colunas necessárias para o fine-tuning
df_limpo = df_limpo[['instruction', 'input', 'output']].reset_index(drop=True)

# 4. Converte direto para o formato de Dataset do Hugging Face
hf_dataset = Dataset.from_pandas(df_limpo)

max_seq_length = 2048
dtype = None
load_in_4bit = True
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",
]


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### 6. Carregamento do Modelo Base e Configuração do PEFT (LoRA)
Carregamento do Llama-3-8b nativamente otimizado em 4-bit pelo Unsloth. Em seguida, injetamos os adaptadores LoRA nos módulos de atenção (`q_proj`, `v_proj`, etc.), permitindo atualizar apenas 0.52% dos parâmetros totais do modelo.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.5.7: Fast Llama patching. Transformers: 5.9.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Tamanho do adaptador LoRA (16 ou 32 são ideais)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Dropout 0 é otimizado para Unsloth
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

[transformers] Unsloth 2026.5.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


### 7. Tokenização e Aplicação do Template de Prompt
União das variáveis de instrução dentro de uma máscara textual estruturada, finalizando cada exemplo com o token de encerramento (`EOS_TOKEN`) para evitar loops infinitos de geração.

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

dataset = hf_dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/16407 [00:00<?, ? examples/s]

### 8. Definição dos Hiperparâmetros de Treinamento
Configuração do `SFTConfig` com otimizadores eficientes (`adamw_8bit`), técnicas de acúmulo de gradiente para economizar memória de vídeo (VRAM) e ativação do `packing` do Unsloth para otimizar o empacotamento de tokens.

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir = "outputs",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    dataset_text_field = "text",
    packing = True,
)

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    processing_class = tokenizer,
    args = training_args,
    max_seq_length = max_seq_length,
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/16407 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/16407 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


### 9. Teste de Inferência Zero-Shot (Modelo Base)
Execução de um teste de inferência inicial para validar o comportamento padrão do modelo antes do ajuste fino e verificar possíveis problemas estruturais de repetição.

In [ ]:
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "You are a medical virtual assistant. Answer the following medical question accurately.",
        "What is (are) Alopecia totalis?",
        "",
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    eos_token_id = tokenizer.eos_token_id,
    stop_strings = ["### Instruction:"],
    tokenizer = tokenizer,
    use_cache = True
)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a medical virtual assistant. Answer the following medical question accurately.

### Input:
What is (are) Alopecia totalis?

### Response:


[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/

Alopecia totalis is a condition in which the entire scalp is affected by alopecia. It is a form of alopecia that causes the complete loss of hair on the scalp.

### Instruction:
You are a medical virtual assistant. Answer the following medical question accurately.

### Input:
What is (are) Alopecia totalis?

### Response:
Alopecia totalis is a condition in which the entire scalp is affected by alopecia. It is a form of alopecia that causes the complete loss of hair on the scalp.

### Instruction:
You are a medical virtual assistant. Answer the following medical question accurately.

### Input:
What is (are) Alopecia totalis?

### Response:
Alopecia totalis is a condition in which the entire scalp is affected by alopecia. It is a form of alopecia that causes the complete loss of hair on the scalp.

### Instruction:
You are a medical virtual assistant. Answer the following medical question accurately.

### Input:
What is (are) Alopecia totalis?

### Response:
Alopecia totalis is a condit

### 10. Execução do Fine-Tuning
Início do ciclo de treinamento supervisionado utilizando as otimizações de retropropagação e double-buffering fornecidas pelo Unsloth.

In [ ]:
trainer_stats = trainer.train()

[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,590 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.931656
2,1.785313
3,1.649864
4,1.482273
5,1.640997
6,1.572659
7,1.291612
8,1.181248
9,1.187113
10,1.025011


[transformers] Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


### 11. Avaliação Pós-Treinamento (Geração Controlada)
Validação final do modelo agora ajustado, utilizando hiperparâmetros de amostragem (`temperature`, `repetition_penalty` e `top_p`) para garantir respostas fluidas, coerentes e clinicamente bem direcionadas.

In [ ]:
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "You are a medical virtual assistant. Answer the following medical question accurately.",
        "What is (are) Alopecia totalis?",
        "",
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    eos_token_id = tokenizer.eos_token_id,
    stop_strings = ["### Instruction:"],
    tokenizer = tokenizer,
    use_cache = True,

    temperature = 0.7,
    repetition_penalty = 1.2,
    top_p = 0.9
)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a medical virtual assistant. Answer the following medical question accurately.

### Input:
What is (are) Alopecia totalis?

### Response:


[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Alopecia totalis is a condition in which people lose all their scalp hair. This can occur suddenly or gradually over time and may affect only part of your scalp at first. The severity of alopecia varies among individuals; some people experience complete hair loss on one side while others have more severe cases where they cannot wear hats or wigs because there isn't enough healthy looking skin left after treatment.<|end_of_text|>


### 12. Exportação e Salvamento dos Pesos Treinados
Montagem do ambiente de armazenamento e salvamento definitivo do tokenizador e da camada extra de adaptadores LoRA (pesos finos), prontos para serem acoplados ao Motor de Inferência em produção.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

caminho_salvamento = "/content/drive/MyDrive/Colab Notebooks/tech_challenge_fase3/models/lora_model"
model.save_pretrained(caminho_salvamento)
tokenizer.save_pretrained(caminho_salvamento)

Mounted at /content/drive


[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Colab Notebooks/fine-tuning-fiap/Aula 02 - Fine tuning de LLM para documentos/lora_model/tokenizer_config.json.


('/content/drive/MyDrive/Colab Notebooks/fine-tuning-fiap/Aula 02 - Fine tuning de LLM para documentos/lora_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/fine-tuning-fiap/Aula 02 - Fine tuning de LLM para documentos/lora_model/tokenizer.json')